# 00 · Git routine: setup and weekly update
**Data Mining · HTW Berlin · Tutorial repository**

This notebook does the Git work for you, step by step. Each cell runs the Git commands, checks the result and tells you what to do if something is wrong. You do not need to type anything in a terminal.

| Part | When | What it does |
|---|---|---|
| 1 · Setup | **once**, the first time | checks Git, your name, your clone, connects the tutorial repo, creates `my_work/` |
| 2 · Weekly update | **every week**, at the start of the tutorial | saves your work, gets the new notebook, copies it into `my_work/` |
| 3 · Cheat sheet | when you want to do it by hand | the commands behind the cells |

Run the cells **from top to bottom**. Every result line starts with `OK`, `WARNING` or `PROBLEM`. If you see `PROBLEM`, fix it before you continue.

You do **not** need to save this notebook. Close it without saving when you are done.

### The three copies of the course folder

```
  TUTORIAL REPO on GitHub            (called: upstream)   only the tutor changes it
  talesthury/Advanced_Data_Analytics_Tutorial_2026_2027
        |
        |  Fork (once, on GitHub)
        v
  YOUR FORK on GitHub                (called: origin)     your copy, you change it
  YOUR-USERNAME/Advanced_Data_Analytics_Tutorial_2026_2027
        |
        |  Clone (once)
        v
  YOUR LAPTOP                        (called: local)      the folder open in VS Code
```

New notebooks travel **upstream → your laptop**. Your own work travels **your laptop → origin**. You cannot break the tutorial repo.

### The golden rule
**Never fill in the original notebooks. Work on a copy inside `my_work/`.**
When the tutor fixes a notebook you have edited, Git cannot decide which version to keep and the update fails. Files in `my_work/` belong only to you, so they never cause problems. Part 2 of this notebook makes the copies for you.

## 0 · Helpers (just run)

In [ ]:
import subprocess, shutil
from pathlib import Path

TUTORIAL_OWNER = "talesthury"
TUTORIAL_REPO = "https://github.com/talesthury/Advanced_Data_Analytics_Tutorial_2026_2027.git"
WORK_DIR = "my_work"
THIS_NOTEBOOK = "00_git_routine.ipynb"


def git(*args, cwd=None, show=True):
    """Run one git command and return (ok, output)."""
    try:
        r = subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd or ROOT)
    except FileNotFoundError:
        return False, "git is not installed"
    out = (r.stdout + r.stderr).strip()
    if show and out:
        print("   " + out.replace("\n", "\n   "))
    return r.returncode == 0, out


def find_root():
    try:
        r = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
    except FileNotFoundError:
        return None
    return Path(r.stdout.strip()) if r.returncode == 0 else None


ROOT = find_root()
print("helpers ready")

---
# Part 1 · Setup (once)

Before this notebook can help, you need the folder on your laptop. That part is in the **README** of the repo:
fork the repo on GitHub, clone **your fork**, open the folder in VS Code. If you can read this inside VS Code, that is done.

### 1.1 · Is Git installed?

In [ ]:
if shutil.which("git"):
    ok, out = git("--version", cwd=".", show=False)
    print("OK      ", out)
else:
    print("PROBLEM  Git is not installed.")
    print("         Install it from https://git-scm.com/downloads (keep the default options),")
    print("         then CLOSE and REOPEN VS Code and run this notebook again.")

### 1.2 · Tell Git who you are
Git writes your name on every save (*commit*). Fill in your name and the **e-mail of your GitHub account**, then run the cell. If you have done this before, the cell just shows what is already set.

In [ ]:
MY_NAME = ""          # e.g. "Anna Schmidt"
MY_EMAIL = ""         # the e-mail of your GitHub account

if MY_NAME and MY_EMAIL:
    git("config", "--global", "user.name", MY_NAME, cwd=".")
    git("config", "--global", "user.email", MY_EMAIL, cwd=".")

# how to combine your work with new files when you pull (prevents an error in newer Git versions)
git("config", "--global", "pull.rebase", "false", cwd=".")

_, name = git("config", "--global", "user.name", cwd=".", show=False)
_, email = git("config", "--global", "user.email", cwd=".", show=False)
if name and email:
    print(f"OK       Git knows you as: {name} <{email}>")
else:
    print("PROBLEM  Name or e-mail missing. Fill in MY_NAME and MY_EMAIL above and run the cell again.")

### 1.3 · Is the clone in a good place?
This cell checks the most common clone mistakes:
- the notebook is not inside a Git folder at all (downloaded as a ZIP instead of cloned)
- the folder is inside **OneDrive, iCloud or Dropbox** (they can damage the hidden `.git` folder)
- the folder is **inside another repo**, for example inside the lecture repo

In [ ]:
ROOT = find_root()
if ROOT is None:
    print("PROBLEM  This folder is not a Git repository.")
    print("         You probably downloaded a ZIP. Follow the README: fork, then clone your fork.")
else:
    print(f"OK       Course folder: {ROOT}")

    if any(w in str(ROOT).lower() for w in ["onedrive", "icloud", "dropbox"]):
        print("WARNING  The folder is inside a cloud-synced folder (OneDrive / iCloud / Dropbox).")
        print("         Better: clone again into e.g. C:\\Users\\you\\Documents\\DataMining (not synced).")
    else:
        print("OK       Not inside OneDrive / iCloud / Dropbox.")

    parent_is_repo, parent = git("rev-parse", "--show-toplevel", cwd=ROOT.parent, show=False)
    if parent_is_repo:
        print(f"WARNING  This repo is INSIDE another repo: {parent}")
        print("         Keep the course repos SIDE BY SIDE, e.g.:")
        print("            DataMining/lecture-repo/")
        print("            DataMining/Advanced_Data_Analytics_Tutorial_2026_2027/")
        print("         Move this folder out (close VS Code first), then reopen it.")
    else:
        print("OK       Not inside another repo.")

### 1.4 · Did you clone **your fork**?
`origin` must point to **your** GitHub account. If you cloned the tutorial repo directly, you cannot save your work on GitHub. No need to clone again: fork the repo on GitHub, paste the link of your fork below and run the cell.

In [ ]:
MY_FORK_URL = ""      # only needed if this cell says PROBLEM, e.g. "https://github.com/YOUR-USERNAME/Advanced_Data_Analytics_Tutorial_2026_2027.git"

if MY_FORK_URL:
    git("remote", "set-url", "origin", MY_FORK_URL)

ok, origin = git("remote", "get-url", "origin", show=False)
if not ok:
    print("PROBLEM  There is no 'origin'. Clone your fork again as described in the README.")
elif f"/{TUTORIAL_OWNER}/" in origin.replace(":", "/"):
    print(f"PROBLEM  origin points to the TUTORIAL repo: {origin}")
    print("         1. On GitHub, open the tutorial repo and click Fork -> Create fork.")
    print("         2. On your fork, click Code and copy the HTTPS link.")
    print("         3. Paste it into MY_FORK_URL above and run this cell again.")
else:
    print(f"OK       origin is your fork: {origin}")

### 1.5 · Connect the tutorial repo (`upstream`)
Your laptop knows your fork. Now it also learns where the new notebooks come from.

In [ ]:
ok, current = git("remote", "get-url", "upstream", show=False)
if not ok:
    git("remote", "add", "upstream", TUTORIAL_REPO)
    print("OK       upstream added.")
elif current.rstrip("/").removesuffix(".git") != TUTORIAL_REPO.removesuffix(".git"):
    git("remote", "set-url", "upstream", TUTORIAL_REPO)
    print("OK       upstream corrected.")
else:
    print("OK       upstream already connected.")

print("\nYour connections:")
git("remote", "-v")

### 1.6 · Create your work folder

In [ ]:
work = ROOT / WORK_DIR
work.mkdir(exist_ok=True)
keep = work / ".gitkeep"
if not keep.exists():
    keep.write_text("")
print(f"OK       Your work folder: {work}")

### 1.7 · Setup summary
Run this cell at the end. If everything says `OK`, the setup is done and you only need Part 2 from now on.

In [ ]:
ROOT = find_root()
checks = {}
checks["Git installed"] = shutil.which("git") is not None
checks["Name and e-mail set"] = bool(git("config", "--global", "user.name", cwd=".", show=False)[1]) and \
                                bool(git("config", "--global", "user.email", cwd=".", show=False)[1])
checks["Inside a cloned repo"] = ROOT is not None
if ROOT is not None:
    _, origin = git("remote", "get-url", "origin", show=False)
    checks["origin is your fork"] = bool(origin) and f"/{TUTORIAL_OWNER}/" not in origin.replace(":", "/")
    checks["upstream connected"] = git("remote", "get-url", "upstream", show=False)[0]
    checks["my_work/ exists"] = (ROOT / WORK_DIR).is_dir()

for name, passed in checks.items():
    print(f"{'OK      ' if passed else 'PROBLEM '} {name}")
print("\nSetup complete." if all(checks.values()) else "\nFix the PROBLEM lines (see the cells above), then run this cell again.")

---
# Part 2 · Weekly update (every week)

Run the four cells **in this order**: save, check, update, copy.

### 2.1 · Save your own work
Everything in `my_work/` is saved (*committed*) and sent to your fork on GitHub (*pushed*). Your work is now safe, even if your laptop breaks.

You can change the message if you want.

In [ ]:
MESSAGE = "My work, week update"

ROOT = find_root()
git("add", WORK_DIR, show=False)
_, staged = git("diff", "--cached", "--name-only", show=False)
if staged:
    git("commit", "-m", MESSAGE, show=False)
    print("OK       Saved locally:")
    print("            " + staged.replace("\n", "\n            "))
else:
    print("OK       Nothing new in my_work/ to save.")

ok, out = git("push", "origin", "HEAD", show=False)
if ok:
    print("OK       Your fork on GitHub is up to date.")
else:
    print("PROBLEM  Could not send your work to GitHub:")
    print("   " + out.replace("\n", "\n   "))
    if "rejected" in out:
        print("         Your fork has changes your laptop does not have. Run 2.3 first, then this cell again.")
    else:
        print("         If a browser window opened, log in to GitHub and run this cell again.")

### 2.2 · Check the golden rule
This cell looks for original notebooks that you edited by mistake. If it finds some, it **copies your version into `my_work/`** (nothing is lost) and **puts the original back**, so the update in 2.3 can work.

In [ ]:
ROOT = find_root()
_, unstaged = git("-c", "core.quotepath=false", "diff", "--name-only", show=False)
_, staged = git("-c", "core.quotepath=false", "diff", "--cached", "--name-only", show=False)

edited = sorted({p.strip() for p in (unstaged + "\n" + staged).splitlines()
                 if p.strip() and not p.strip().startswith(WORK_DIR + "/")})

if not edited:
    print("OK       No original files were edited.")
for path in edited:
    src = ROOT / path
    if path.endswith(THIS_NOTEBOOK):
        git("restore", "--staged", "--worktree", "--", path, show=False)
        print(f"OK       {path} was saved with outputs, reset it (outputs on screen may disappear).")
        continue
    dst = ROOT / WORK_DIR / f"{src.stem}_RESCUED{src.suffix}"
    n = 1
    while dst.exists():
        n += 1
        dst = ROOT / WORK_DIR / f"{src.stem}_RESCUED{n}{src.suffix}"
    shutil.copy2(src, dst)
    git("restore", "--staged", "--worktree", "--", path, show=False)
    print(f"WARNING  You edited {path}.")
    print(f"         Your version is now in {dst.relative_to(ROOT)}, the original was put back.")

### 2.3 · Get the new notebooks
New files come from the tutorial repo to your laptop, and your fork on GitHub is updated too.

> Alternative without this notebook: on your fork on GitHub click **Sync fork → Update branch**, then in VS Code click **Sync Changes** in the Source Control panel.

In [ ]:
ROOT = find_root()
before = set(p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.ipynb"))

ok, out = git("pull", "--no-edit", "upstream", "main", show=False)
if ok:
    print("OK       Up to date with the tutorial repo.")
    git("push", "origin", "HEAD", show=False)
    after = set(p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.ipynb"))
    new = sorted(p for p in after - before if not p.startswith(WORK_DIR + "/"))
    print("         New notebooks: " + (", ".join(new) if new else "none"))
elif "CONFLICT" in out:
    _, conflicted = git("-c", "core.quotepath=false", "diff", "--name-only", "--diff-filter=U", show=False)
    conflicted = [f.strip() for f in conflicted.splitlines() if f.strip()]
    if not conflicted or any(f.startswith(WORK_DIR + "/") for f in conflicted):
        git("merge", "--abort", show=False)
        print("PROBLEM  Conflict the notebook cannot solve alone. The update was cancelled, nothing is lost.")
        print("         Bring this message to the tutorial: " + ", ".join(conflicted))
    else:
        for path in conflicted:
            src = ROOT / path
            dst = ROOT / WORK_DIR / f"{src.stem}_RESCUED{src.suffix}"
            n = 1
            while dst.exists():
                n += 1
                dst = ROOT / WORK_DIR / f"{src.stem}_RESCUED{n}{src.suffix}"
            mine = subprocess.run(["git", "show", f"HEAD:{path}"], capture_output=True, cwd=ROOT)
            if mine.returncode == 0:
                dst.write_bytes(mine.stdout)
            if not git("checkout", "--theirs", "--", path, show=False)[0]:
                git("rm", "-q", "--", path, show=False)
            else:
                git("add", "--", path, show=False)
            print(f"WARNING  You had saved changes in the original {path}.")
            if mine.returncode == 0:
                print(f"         Your version is now in {dst.relative_to(ROOT).as_posix()}, the tutor's version was taken.")
        git("add", "--", WORK_DIR, show=False)
        git("commit", "--no-edit", "-q", show=False)
        git("push", "origin", "HEAD", show=False)
        print("OK       Update finished. Remember the golden rule: work only in my_work/.")
elif "would be overwritten" in out:
    print("PROBLEM  You have unsaved edits in an original file. Run 2.2, then this cell again.")
elif "Could not resolve host" in out or "unable to access" in out:
    print("PROBLEM  No connection to GitHub. Check your internet (eduroam / VPN) and try again.")
else:
    print("PROBLEM  The update did not work:")
    print("   " + out.replace("\n", "\n   "))

### 2.4 · Copy this week's notebook into `my_work/`
The first part lists every tutorial notebook and shows whether you already have a copy. Then write the one you want into `NOTEBOOK` and run the cell again. Existing copies are **never** overwritten.

In [ ]:
NOTEBOOK = ""     # e.g. "02_outliers/02_outliers_tutorial.ipynb"  (copy it from the list below)

ROOT = find_root()
originals = sorted(p for p in ROOT.rglob("*.ipynb")
                   if WORK_DIR not in p.relative_to(ROOT).parts
                   and ".ipynb_checkpoints" not in p.parts
                   and p.name != THIS_NOTEBOOK)

print("Tutorial notebooks:")
for p in originals:
    has_copy = (ROOT / WORK_DIR / f"{p.stem}_MINE{p.suffix}").exists()
    print(f"   {'[copied]' if has_copy else '[      ]'}  {p.relative_to(ROOT).as_posix()}")

if NOTEBOOK:
    src = ROOT / NOTEBOOK
    dst = ROOT / WORK_DIR / f"{src.stem}_MINE{src.suffix}"
    if not src.exists():
        print(f"\nPROBLEM  {NOTEBOOK} does not exist. Copy the path exactly from the list.")
    elif dst.exists():
        print(f"\nOK       You already have {dst.relative_to(ROOT).as_posix()}. Open that one.")
    else:
        shutil.copy2(src, dst)
        print(f"\nOK       Created {dst.relative_to(ROOT).as_posix()}. Open it and start working.")

**Done for this week.** Open your copy in `my_work/` and work there. Next week, start again at 2.1.

> **Note:** data files that a notebook reads (for example a `.csv` in the same folder) are not copied. If a notebook in `my_work/` cannot find a file, use the path `../02_outliers/data.csv` instead of `data.csv`.

---
# Part 3 · Cheat sheet: the commands behind the cells

Run these in a terminal inside the course folder (VS Code: *Terminal → New Terminal*).

| Cell | What it does | Command |
|---|---|---|
| 1.2 | set your name / e-mail | `git config --global user.name "Your Name"` · `git config --global user.email "you@example.com"` |
| 1.4 | point origin to your fork | `git remote set-url origin https://github.com/YOU/Advanced_Data_Analytics_Tutorial_2026_2027.git` |
| 1.5 | connect the tutorial repo | `git remote add upstream https://github.com/talesthury/Advanced_Data_Analytics_Tutorial_2026_2027.git` |
| 1.5 | show connections | `git remote -v` |
| 2.1 | save your work | `git add my_work` · `git commit -m "message"` · `git push` |
| 2.2 | see what changed | `git status` |
| 2.2 | put an original file back | `git restore path/to/file.ipynb` |
| 2.3 | get new notebooks | `git pull upstream main` · `git push` |
| 2.3 | cancel a failed update | `git merge --abort` |

### Common messages
| Message | Meaning | Fix |
|---|---|---|
| `fatal: not a git repository` | the terminal is in the wrong folder | `cd` into the course folder |
| `destination path ... already exists` | you already cloned it | open that folder instead of cloning again |
| `Your local changes ... would be overwritten` | you edited an original file | run cell 2.2 |
| `CONFLICT (content)` | same file changed by you and the tutor | cell 2.3 saves your version in `my_work/` and takes the tutor's; by hand: `git merge --abort` |
| `! [rejected] ... (fetch first)` | your fork has changes your laptop does not | pull first (2.3), then push (2.1) |
| `Need to specify how to reconcile divergent branches` | a Git setting is missing | `git config --global pull.rebase false` (cell 1.2 does it) |

**Still stuck?** Bring the full error message to the tutorial. Do **not** delete the folder and clone again before asking: your work in `my_work/` might only exist on your laptop.